In [19]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.utils import resample
from xgboost import DMatrix, XGBClassifier
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import Perceptron
from sklearn.metrics import confusion_matrix, accuracy_score, ConfusionMatrixDisplay
from sklearn.neural_network import MLPRegressor

## 1. Utilisation AdaBoost
import des données MNIST

In [ ]:
X_mnist, y_mnist = fetch_openml('mnist_784', return_X_y=True, as_frame=False)
y_mnist = y_mnist.astype(np.int8)  # Convert labels to integers
X_mnist = X_mnist / 255.0

X_train, X_test, y_train, y_test = train_test_split(X_mnist, y_mnist, test_size=0.2, random_state=42, stratify= y_mnist)

découpage des données pour plusieurs taches de reconnaissance :
- 3 vs 8
- 0-4 vs 5-9
- 0 vs 1 vs 2 vs ... vs 9

In [16]:
def make_versus(classes_a, classes_b, N_train=None, N_test=None):
    '''
    créer les sous-listes contennant les individus des groupes de classes a et b
    label 0 : classes_a
    label 1 : classes_b
    '''
    def sel(Xs, ys, n):
        if n == None:
            n=-1
        m = np.isin(ys, classes_a + classes_b)
        Xs, ys = Xs[m][:n], ys[m][:n]
        return Xs, np.isin(ys, classes_b).astype(int)
    X_tr, y_tr = sel(X_train, y_train, N_train)
    X_te, y_te = sel(X_test, y_test, N_test)
    return X_tr, X_te, y_tr, y_te

VERSUS = {
    "3 vs 8": make_versus([3], [8]),
    "0-4 vs 5-9": make_versus([0,1,2,3,4], [5,6,7,8,9]),
    "versus all number": [X_train, X_test, y_train, y_test]
}

for title, (X_tr, X_te, y_tr, y_te) in VERSUS.items():
    print(title, "| train :", X_tr.shape, "| test :", X_te.shape)

3 vs 8 | train : (11172, 784) | test : (2792, 784)
0-4 vs 5-9 | train : (55999, 784) | test : (13999, 784)
versus all number | train : (56000, 784) | test : (14000, 784)


Choisir plusieurs paramètre de AdaBoost : 
- Learning rate
- Choisir le classifieur faibles : 
  - arbres avec profondeur variable
  - perceptron
- nombre d'estimateurs/de classifieur d'affilé (étapes de boosting)    

In [ ]:
etapes = [10, 50, 200]

LR = [0.01, 0.1, 0.5, 1]
max_depth = [1, 3, 8, 20, 50]   # hyperparamètre du classifieur faible (max_depth pour l'arbre, max_iter pour le perceptron)
classif_faible = {
    "arbre":      lambda p: DecisionTreeClassifier(max_depth=p),
    "perceptron": lambda p: Perceptron(max_iter=p, tol=None, random_state=0)
}

def make_ada(classif_faible, n, lr=0.1):
    return AdaBoostClassifier(estimator=classif_faible, n_estimators=n, learning_rate=lr, random_state=0)

def entrainer(tasks):
    '''
    entrainer notre adaboost pour chaque (tache, classifieur faible, max_depth, lr)
    un seul fit avec max(etapes) estimators, puis staged_predict pour lire les prédictions à chaque étape
    preds[(tache, nom, p, lr, s)] = prédictions sur le test
    '''
    models, preds = {}, {}
    for tache, (X_tr, X_te, y_tr, y_te) in tasks.items():
        for nom, f in classif_faible.items():
            for p in max_depth:
                for lr in LR:
                    ab = make_ada(f(p), max(etapes), lr).fit(X_tr, y_tr)
                    models[(tache, nom, p, lr)] = ab
                    save = list(ab.staged_predict(X_te))
                    for s in etapes:
                        preds[(tache, nom, p, lr, s)] = save[min(s, len(save)) - 1]
    return models, preds

def afficher_mc(tasks, preds, p=max_depth[0], lr=LR[0], titre=""):
    '''matrices de confusion pour un (max_depth, lr) fixé : lignes = classif faible, colonnes = étapes'''
    rows = []
    for tache, (X_tr, X_te, y_tr, y_te) in tasks.items():
        fig, axes = plt.subplots(len(classif_faible), len(etapes), figsize=(5*len(etapes), 5*len(classif_faible)), constrained_layout=True, squeeze=False)
        for i, nom in enumerate(classif_faible):
            for j, s in enumerate(etapes):
                y_pred = preds[(tache, nom, p, lr, s)]
                acc = accuracy_score(y_te, y_pred)
                rows.append({"tache": tache, "classifieur faible": nom, "étapes": s, "accuracy": acc})
                ConfusionMatrixDisplay(confusion_matrix(y_te, y_pred)).plot(ax=axes[i,j], colorbar=False, cmap="Blues")
                axes[i,j].set_title(f"{nom}, {s} etapes \n acc = {acc:.3f}")
        fig.suptitle(f"Matrices de confusion - {tache} (max_depth={p}, lr={lr}) {titre}")
        plt.show()
    df = pd.DataFrame(rows)
    return df.pivot_table(index=["tache", "classifieur faible"], columns="étapes", values="accuracy").round(3)

def afficher_acc(tasks, preds, titre=""):
    '''
    une figure par tache : lignes = classif faible, colonnes = n_estimators (etapes)
    chaque case = matrice d'accuracy (axe y : max_depth, axe x : learning rate)
    '''
    for tache, (X_tr, X_te, y_tr, y_te) in tasks.items():
        acc = {k: accuracy_score(y_te, v) for k, v in preds.items() if k[0] == tache}
        vmin, vmax = min(acc.values()), max(acc.values())
        fig, axes = plt.subplots(len(classif_faible), len(etapes), figsize=(5*len(etapes), 4*len(classif_faible)), constrained_layout=True, squeeze=False)
        for i, nom in enumerate(classif_faible):
            for j, s in enumerate(etapes):
                M = np.array([[acc[(tache, nom, p, lr, s)] for lr in LR] for p in max_depth])
                ax = axes[i,j]
                im = ax.imshow(M, cmap="viridis", vmin=vmin, vmax=vmax, aspect="auto")
                ax.set_xticks(range(len(LR)), LR)
                ax.set_yticks(range(len(max_depth)), max_depth)
                ax.set_xlabel("learning rate")
                ax.set_ylabel("max_depth")
                ax.set_title(f"{nom}, n_estimators = {s}")
                for a in range(len(max_depth)):
                    for b in range(len(LR)):
                        ax.text(b, a, f"{M[a,b]:.3f}", ha="center", va="center", fontsize=8,
                                color="white" if M[a,b] < (vmin+vmax)/2 else "black")
        fig.colorbar(im, ax=axes, shrink=0.6, label="accuracy")
        fig.suptitle(f"Accuracy - {tache} {titre}")
        plt.show()


In [ ]:
models, preds = entrainer(VERSUS)
afficher_mc(VERSUS, preds)
afficher_acc(VERSUS, preds)